# FraudShield — Exploratory Data Analysis

Executable student research notebook for the **original user-supplied Kaggle CSV**. Dataset is not recreated, simulated, or fabricated. Place the source file in `data/creditcard_2023.csv` before running.

**Important:** This dataset is 50/50 class-balanced, has **no timestamps**, no merchants and no location. No historical trend is supported.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt
root = Path.cwd()
if root.name == 'analytics': root=root.parent
path = root/'data/creditcard_2023.csv'
if not path.is_file(): raise FileNotFoundError('Download original Kaggle CSV to data/creditcard_2023.csv')
df=pd.read_csv(path)
print('Real uploaded dataset:',df.shape)
df.head()

## 1. Schema and validation

In [ ]:
features=[f'V{i}' for i in range(1,29)]+['Amount']
assert list(df.columns)==['id',*features,'Class'] and len(df)==568630
print('Class counts:',df.Class.value_counts().to_dict())
print('Missing cells:',df.isna().sum().sum())
print('Duplicate IDs:',df.id.duplicated().sum())
print('Repeated feature vectors:',df.duplicated(subset=features).sum())
print(df.Amount.describe())

## 2. Label balance
Exactly 50% label 1; this is a **dataset composition** statistic, not a bank fraud prevalence estimate.

In [ ]:
counts=df.Class.value_counts().sort_index()
fig,ax=plt.subplots(figsize=(7,4))
ax.bar(['Legitimate / 0','Fraud-labelled / 1'],counts.values)
ax.set_title('Artificially balanced source labels')
ax.set_ylabel('Number of records')
plt.tight_layout();plt.show()

## 3. Amount distributions
Recorded amount currency unspecified. No temporal grouping possible.

In [ ]:
fig,ax=plt.subplots(figsize=(10,4))
for label, group in df.groupby('Class'):
    ax.hist(group.Amount,bins=35,alpha=.45,label=f'Known label {label}')
ax.set_xlabel('Recorded amount (unverified currency)')
ax.set_ylabel('Record count')
ax.set_title('Amount by known class')
ax.legend();plt.tight_layout();plt.show()
print(df.groupby('Class').Amount.agg(['count','mean','median','sum']).round(2))

## 4. Strongest anonymous feature-label associations
Pearson correlation does not prove causality. The meaning of V1–V28 is unknown.

In [ ]:
correlations=df[features+['Class']].corr(numeric_only=True)['Class'].drop('Class')
top=correlations.reindex(correlations.abs().sort_values(ascending=False).index).head(12)
fig,ax=plt.subplots(figsize=(9,5))
ax.barh(top.index[::-1],top.values[::-1])
ax.set_xlabel('Pearson correlation (signed)')
ax.set_title('Top anonymous predictor relationships')
plt.tight_layout();plt.show()
top

## 5. Business segmentation by amount band
Reconciles with SQL views and the Analytics dashboard; not a time-series chart.

In [ ]:
bands=pd.cut(df.Amount,[0,5000,10000,15000,20000,np.inf],labels=['0–5k','5–10k','10–15k','15–20k','20k+'])
summary=df.assign(amount_band=bands).groupby('amount_band',observed=True).Class.agg(['count','sum','mean'])
assert summary['count'].sum()==568630
print(summary)

## 6. Honest independent model comparison
The canonical model training script is `scripts/build_data.py`, with fixed random state, no ID predictors, exact duplicate exclusion and validation-only model selection.

In [ ]:
import json
report=json.loads((root/'analytics/summary.json').read_text())
print('Selected model:',report['selected_model'])
for name,item in report['models'].items():
 t=item['test']
 print(name,'F1=',round(t['f1'],5),'PR-AUC=',round(t['pr_auc'],5),'TP/TN/FP/FN=',tuple(t[k] for k in ['tp','tn','fp','fn']))

## Limitations
The high test F1 is not a real banking performance claim. The source dataset is artificially balanced and provenance of the synthetic construction is not fully verified. Near-duplicate source-generation leakage, real-life drift and calibration require additional validation on independent dated naturally imbalanced data. Simulated scenario costs are not verified savings.